# 09 — Comparaison de stratégies lag 1

Ce notebook compare plusieurs règles de portefeuille avec des prédictions walk-forward hors échantillon. Le lag est strict : la prédiction de J-1 est appliquée au gap de J. Les stratégies sont comparées à coûts de 5, 10 et 20 bp. Le test final ne sert pas à choisir les seuils : les paramètres doivent être fixés avant l'interprétation.

In [14]:
import os, json
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

PROJECT = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project"
DATA = os.path.join(PROJECT, "data", "processed")
df = pd.read_csv(os.path.join(DATA, "DATASET_MODELISATION_2020_2022.csv"))
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
with open(os.path.join(DATA, "config_modelisation.json"), encoding="utf-8") as f:
    cfg = json.load(f)
features = [c for c in cfg["features"] if c in df.columns]
purge = cfg["purge_jours"]

def logit():
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("std", StandardScaler()), ("clf", LogisticRegression(max_iter=2000))])

def folds(dates, n=8, test_size=60, min_train=250):
    days = np.sort(pd.unique(dates)); result = []; end = len(days)
    for _ in range(n):
        split = end - test_size
        if split - purge < min_train: break
        result.append((days[:split-purge], days[split:end])); end = split
    return result[::-1]

d = df.dropna(subset=["y_gap", "gap"]).copy()
blocks = []
for train_days, test_days in folds(d["Date"]):
    train = d[d["Date"].isin(train_days)]
    test = d[d["Date"].isin(test_days)].copy()
    if len(train) < 250 or test["y_gap"].nunique() < 2:
        continue
    model = clone(logit()).fit(train[features], train["y_gap"])
    test["proba"] = model.predict_proba(test[features])[:, 1]
    blocks.append(test)
p = pd.concat(blocks).sort_values(["Ticker", "Date"]).reset_index(drop=True)
p["proba_oos"] = p["proba"]
p_full = p.copy()
p["proba_lag1"] = p.groupby("Ticker")["proba_oos"].shift(1)
p = p.dropna(subset=["proba_lag1"]).copy()
p["proba"] = p["proba_lag1"]
print(f"Prédictions lag 1 : {len(p):,} lignes, {p['Date'].nunique()} jours")
print(f"AUC descriptive du signal décalé : {roc_auc_score(p['y_gap'], p['proba']):.4f}")

Prédictions lag 1 : 1,195 lignes, 239 jours
AUC descriptive du signal décalé : 0.5309


## Stratégies testées

Les seuils sont fixés à l'avance : 0,55/0,45 et 0,60/0,40. La stratégie long-only ne prend jamais de position short. Les poids sont normalisés par jour afin que le portefeuille ait une exposition comparable entre les journées.

In [15]:
def make_positions(data, name):
    x = data.copy()
    e = x["proba"] - 0.5
    if name == "long_only_055":
        raw = (x["proba"] > 0.55).astype(float)
    elif name == "extreme_ls_060":
        raw = np.where(x["proba"] > 0.60, 1.0, np.where(x["proba"] < 0.40, -1.0, 0.0))
    elif name == "proportional_ls":
        raw = np.clip(e / 0.15, -1.0, 1.0)
    elif name == "risk_scaled_ls":
        vol = x["vol_20"].replace(0, np.nan).fillna(x["vol_20"].median())
        raw = np.clip(e / 0.15, -1.0, 1.0) * (0.02 / vol).clip(0.25, 3.0)
    elif name == "long_only_extreme":
        raw = (x["proba"] > 0.60).astype(float)
    else:
        raise ValueError(name)
    x["raw_position"] = raw
    def normalize(group):
        gross = group["raw_position"].abs().sum()
        group["position"] = group["raw_position"] / gross if gross > 0 else 0.0
        return group
    return x.groupby("Date", group_keys=False).apply(normalize)

strategies = ["long_only_055", "long_only_extreme", "extreme_ls_060", "proportional_ls", "risk_scaled_ls"]

def evaluate(data, cost_bp):
    x = data.sort_values(["Ticker", "Date"]).copy()
    x["turnover"] = (x.groupby("Ticker")["position"].diff().abs().fillna(x["position"].abs()))
    x["gross_pnl"] = x["position"] * x["gap"]
    x["cost"] = x["turnover"] * cost_bp / 10000
    x["net_pnl"] = x["gross_pnl"] - x["cost"]
    daily = x.groupby("Date").agg(net=("net_pnl", "sum"), gross=("gross_pnl", "sum"), cost=("cost", "sum"), turnover=("turnover", "sum"), n_pos=("position", lambda s: (s != 0).sum()))
    r = daily["net"]
    equity = (1 + r).cumprod()
    sharpe = r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan
    drawdown = (equity / equity.cummax() - 1).min()
    from scipy.stats import ttest_1samp
    p_value = ttest_1samp(r, 0, nan_policy="omit").pvalue if len(r) > 1 else np.nan
    return {"cost_bp": cost_bp, "annual_return": r.mean() * 252, "sharpe": sharpe, "max_drawdown": drawdown, "win_rate": (r > 0).mean(), "p_value_mean": p_value, "mean_daily_pnl": r.mean(), "total_turnover": daily["turnover"].sum(), "days": len(r), "positions": int(daily["n_pos"].sum())}


In [16]:
rows = []
for strategy in strategies:
    positions = make_positions(p, strategy)
    for cost in [5, 10, 20]:
        row = evaluate(positions, cost)
        row["strategy"] = strategy
        rows.append(row)
results = pd.DataFrame(rows)[["strategy", "cost_bp", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean", "mean_daily_pnl", "total_turnover", "days", "positions"]]
display(results.sort_values(["cost_bp", "sharpe"], ascending=[True, False]).round(4))
out = os.path.join(DATA, "BACKTEST_STRATEGIES_LAG1_RESULTS.csv")
results.to_csv(out, index=False)
print("Export :", out)

C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return x.groupby("Date", group_keys=False).apply(normalize)
C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return x.groupby("Date", group_keys=False).apply(normalize)
C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarn

,strategy,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions
0,long_only_055,5,0.1265,0.7572,-0.1288,0.5063,0.4616,0.0005,239.9000,239,675
3,long_only_extreme,5,0.0972,0.5550,-0.1295,0.4686,0.5893,0.0004,256.7000,239,612
9,proportional_ls,5,-0.0396,-0.2986,-0.1318,0.4979,0.7715,-0.0002,273.5012,239,1195
12,risk_scaled_ls,5,-0.0408,-0.3289,-0.1321,0.5146,0.7490,-0.0002,275.2913,239,1195
6,extreme_ls_060,5,-0.0888,-0.6323,-0.1414,0.4854,0.5387,-0.0004,300.9667,239,946
1,long_only_055,10,0.0001,0.0004,-0.1373,0.4812,0.9997,0.0000,239.9000,239,675
4,long_only_extreme,10,-0.0382,-0.2179,-0.1379,0.4603,0.8321,-0.0002,256.7000,239,612
10,proportional_ls,10,-0.1838,-1.3829,-0.2044,0.4310,0.1793,-0.0007,273.5012,239,1195
13,risk_scaled_ls,10,-0.1859,-1.4966,-0.2044,0.4477,0.1463,-0.0007,275.2913,239,1195
7,extreme_ls_060,10,-0.2474,-1.7605,-0.2524,0.4393,0.0877,-0.0010,300.9667,239,946


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_STRATEGIES_LAG1_RESULTS.csv


## Lecture

La meilleure stratégie ne doit pas être choisie uniquement selon le Sharpe du test. Il faut privilégier une règle fixée avant le test, stable à 5/10/20 bp, avec un drawdown acceptable et un résultat cohérent par sous-période. Un Sharpe positif à 0 bp mais négatif à 10 bp signifie que le signal est trop faible pour être exploité après coûts.

## Analyse des retards de 1 à 5 jours

Chaque retard est calculé à partir de la prédiction walk-forward originale : `proba de J-k` est appliquée au gap de J. Les mêmes stratégies et coûts sont utilisés pour comparer uniquement l'effet du délai.

In [17]:
lag_rows = []
for lag in range(1, 6):
    lag_data = p_full.copy()
    lag_data["proba"] = lag_data.groupby("Ticker")["proba_oos"].shift(lag)
    lag_data = lag_data.dropna(subset=["proba"]).copy()
    for strategy in strategies:
        positions = make_positions(lag_data, strategy)
        for cost in [5, 10, 20]:
            row = evaluate(positions, cost)
            row.update({"lag": lag, "strategy": strategy})
            lag_rows.append(row)
lag_results = pd.DataFrame(lag_rows)
lag_results = lag_results[["lag", "strategy", "cost_bp", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean", "mean_daily_pnl", "total_turnover", "days", "positions"]]
display(lag_results.sort_values(["cost_bp", "sharpe"], ascending=[True, False]).round(4))
lag_out = os.path.join(DATA, "BACKTEST_STRATEGIES_LAG1_TO_LAG5_RESULTS.csv")
lag_results.to_csv(lag_out, index=False)
print("Export :", lag_out)

C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return x.groupby("Date", group_keys=False).apply(normalize)
C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return x.groupby("Date", group_keys=False).apply(normalize)
C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarn

,lag,strategy,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions
0,1,long_only_055,5,0.1265,0.7572,-0.1288,0.5063,0.4616,0.0005,239.9000,239,675
18,2,long_only_extreme,5,0.1279,0.6953,-0.1045,0.5084,0.4999,0.0005,254.7000,238,610
15,2,long_only_055,5,0.1015,0.5746,-0.1206,0.5252,0.5771,0.0004,237.9000,238,672
3,1,long_only_extreme,5,0.0972,0.5550,-0.1295,0.4686,0.5893,0.0004,256.7000,239,612
51,4,extreme_ls_060,5,0.0896,0.5351,-0.1015,0.5042,0.6051,0.0004,297.1333,236,937
...,...,...,...,...,...,...,...,...,...,...,...,...
14,1,risk_scaled_ls,20,-0.4762,-3.8108,-0.3819,0.3640,0.0003,-0.0019,275.2913,239,1195
8,1,extreme_ls_060,20,-0.5648,-3.9987,-0.4348,0.3682,0.0001,-0.0022,300.9667,239,946
44,3,risk_scaled_ls,20,-0.7755,-4.9808,-0.5407,0.3165,0.0000,-0.0031,271.9658,237,1185
41,3,proportional_ls,20,-0.7686,-5.0063,-0.5363,0.3418,0.0000,-0.0031,270.2903,237,1185


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_STRATEGIES_LAG1_TO_LAG5_RESULTS.csv


## Benchmarks, robustesse et significativité

Cette section complète le cahier des charges : benchmark Buy & Hold sur les gaps disponibles, analyse par période et par ticker, et test t de la moyenne des rendements journaliers. Un indice de marché externe doit être ajouté séparément si une série SPY/S&P 500 synchronisée est disponible.

In [18]:
# Benchmark Buy & Hold égal-pondéré sur l'univers observé.
# Il s'agit d'un benchmark sur les gaps (pas d'un rendement close-to-close complet).
benchmark = p.copy()
benchmark["position"] = 1.0 / benchmark.groupby("Date")["Ticker"].transform("nunique")
benchmark_rows = []
for cost in [0, 5, 10, 20]:
    row = evaluate(benchmark, cost)
    row["benchmark"] = "buy_hold_equal_weight_gap"
    benchmark_rows.append(row)
benchmark_results = pd.DataFrame(benchmark_rows)
display(benchmark_results.round(4))
benchmark_out = os.path.join(DATA, "BACKTEST_BENCHMARKS_RESULTS.csv")
benchmark_results.to_csv(benchmark_out, index=False)
print("Export :", benchmark_out)
print("Indice de marché externe : non disponible dans DATASET_MODELISATION_2020_2022.csv")

,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions,benchmark
0,0,0.1134,0.6432,-0.1357,0.59,0.5316,0.0005,1.0,239,1195,buy_hold_equal_weight_gap
1,5,0.1129,0.6401,-0.1357,0.59,0.5336,0.0004,1.0,239,1195,buy_hold_equal_weight_gap
2,10,0.1124,0.6370,-0.1357,0.59,0.5356,0.0004,1.0,239,1195,buy_hold_equal_weight_gap
3,20,0.1113,0.6308,-0.1357,0.59,0.5396,0.0004,1.0,239,1195,buy_hold_equal_weight_gap


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_BENCHMARKS_RESULTS.csv
Indice de marché externe : non disponible dans DATASET_MODELISATION_2020_2022.csv


In [19]:
# Robustesse avec une règle fixée à l'avance : lag 1, long_only_055, coût 10 bp.
fixed_strategy = make_positions(p, "long_only_055")
robust_rows = []
for year, group in fixed_strategy.groupby(fixed_strategy["Date"].dt.year):
    if group["Date"].nunique() > 1:
        row = evaluate(group, 10)
        row.update({"dimension": "year", "group": str(year)})
        robust_rows.append(row)
for ticker, group in fixed_strategy.groupby("Ticker"):
    if group["Date"].nunique() > 1:
        row = evaluate(group, 10)
        row.update({"dimension": "ticker", "group": ticker})
        robust_rows.append(row)
robustness_results = pd.DataFrame(robust_rows)
display(robustness_results[["dimension", "group", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean", "days", "positions"]].round(4))
robust_out = os.path.join(DATA, "BACKTEST_ROBUSTNESS_YEAR_TICKER_RESULTS.csv")
robustness_results.to_csv(robust_out, index=False)
print("Export :", robust_out)

C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return x.groupby("Date", group_keys=False).apply(normalize)


,dimension,group,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,days,positions
0,year,2021,0.0558,0.3976,-0.1153,0.5051,0.7262,196,571
1,year,2022,-0.2523,-0.9745,-0.1373,0.3721,0.6893,43,104
2,ticker,AAPL,-0.0534,-1.6963,-0.0620,0.2552,0.0999,239,132
3,ticker,AMZN,-0.0111,-0.3300,-0.0387,0.2803,0.7482,239,123
4,ticker,META,0.0486,0.8864,-0.0347,0.3305,0.3889,239,135
5,ticker,NVDA,0.0120,0.1721,-0.0417,0.3264,0.8670,239,142
6,ticker,TSLA,0.0039,0.0506,-0.0749,0.3264,0.9607,239,143


Export : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_ROBUSTNESS_YEAR_TICKER_RESULTS.csv


## Benchmark de marché : SPY / S&P 500

SPY est un ETF qui réplique le S&P 500. Pour rester comparable avec le projet, le benchmark utilise le rendement overnight : `Open_t / Close_{t-1} - 1`. Les données sont téléchargées depuis Yahoo Finance et exportées séparément.

In [20]:
import yfinance as yf

spy = yf.download(
    "SPY",
    start="2020-01-01",
    end="2023-01-01",
    auto_adjust=False,
    progress=False,
    threads=False,
)
if spy.empty:
    raise RuntimeError("Yahoo Finance n'a retourné aucune donnée pour SPY.")
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)
spy = spy.reset_index()
spy["Date"] = pd.to_datetime(spy["Date"], utc=True).dt.tz_localize(None)
spy["gap"] = spy["Open"] / spy["Close"].shift(1) - 1
spy_benchmark = spy[["Date", "gap"]].dropna().copy()
spy_benchmark["Ticker"] = "SPY"
spy_benchmark["position"] = 1.0
spy_benchmark = spy_benchmark[spy_benchmark["Date"].isin(p["Date"])].copy()
if spy_benchmark.empty:
    raise RuntimeError("Aucune date SPY ne correspond aux dates du backtest.")

spy_rows = []
for cost in [0, 5, 10, 20]:
    row = evaluate(spy_benchmark, cost)
    row["benchmark"] = "SPY_overnight_gap"
    spy_rows.append(row)
spy_results = pd.DataFrame(spy_rows)
display(spy_results.round(4))
spy_out = os.path.join(DATA, "BACKTEST_SPY_BENCHMARK_RESULTS.csv")
spy_results.to_csv(spy_out, index=False)
print("Export SPY :", spy_out)

strategy_lag1 = make_positions(p, "long_only_055")
comparison_rows = []
for cost in [0, 5, 10, 20]:
    row = evaluate(strategy_lag1, cost)
    row["benchmark"] = "sentiment_lag1_long_only_055"
    comparison_rows.append(row)
comparison = pd.concat([spy_results, pd.DataFrame(comparison_rows)], ignore_index=True)
display(comparison[["benchmark", "cost_bp", "annual_return", "sharpe", "max_drawdown", "win_rate", "p_value_mean"]].round(4))
comparison_out = os.path.join(DATA, "BACKTEST_SPY_VS_SENTIMENT_RESULTS.csv")
comparison.to_csv(comparison_out, index=False)
print("Export comparaison :", comparison_out)

,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean,mean_daily_pnl,total_turnover,days,positions,benchmark
0,0,0.0384,0.4470,-0.0735,0.5941,0.6638,0.0002,1.0,239,239,SPY_overnight_gap
1,5,0.0379,0.4407,-0.0735,0.5941,0.6682,0.0002,1.0,239,239,SPY_overnight_gap
2,10,0.0373,0.4344,-0.0735,0.5941,0.6726,0.0001,1.0,239,239,SPY_overnight_gap
3,20,0.0363,0.4218,-0.0735,0.5941,0.6816,0.0001,1.0,239,239,SPY_overnight_gap


Export SPY : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_SPY_BENCHMARK_RESULTS.csv


C:\Users\semy4\AppData\Local\Temp\ipykernel_6952\255078859.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return x.groupby("Date", group_keys=False).apply(normalize)


,benchmark,cost_bp,annual_return,sharpe,max_drawdown,win_rate,p_value_mean
0,SPY_overnight_gap,0,0.0384,0.4470,-0.0735,0.5941,0.6638
1,SPY_overnight_gap,5,0.0379,0.4407,-0.0735,0.5941,0.6682
2,SPY_overnight_gap,10,0.0373,0.4344,-0.0735,0.5941,0.6726
3,SPY_overnight_gap,20,0.0363,0.4218,-0.0735,0.5941,0.6816
4,sentiment_lag1_long_only_055,0,0.2530,1.5159,-0.1202,0.5314,0.1412
5,sentiment_lag1_long_only_055,5,0.1265,0.7572,-0.1288,0.5063,0.4616
6,sentiment_lag1_long_only_055,10,0.0001,0.0004,-0.1373,0.4812,0.9997
7,sentiment_lag1_long_only_055,20,-0.2529,-1.5028,-0.2671,0.4519,0.1446


Export comparaison : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\BACKTEST_SPY_VS_SENTIMENT_RESULTS.csv
